# Module 2: Recall Layer - Candidate Generation (MBA + ALS)

## Objective
Construct a high-recall candidate set (100-200 items per user) by combining:
1. **Organic Market Basket Analysis (MBA)** for familiarity signal.
2. **Implicit ALS Collaborative Filtering** for discovery signal.

## Methodological Principles
- Use **organic baskets** for MBA to reduce promotion-induced confounding.
- Use **retailer-revenue-weighted confidence** for ALS: $c_{ui} = 1 + \log_{10}(\text{Revenue\_Retailer}_{ui}+1)$.
- Normalize both recall signals to $[0,1]$ for downstream utility integration.
- Enforce initial uplift guardrail by excluding items purchased in the recent 4 weeks.

## Deliverables
- `association_rules.csv` with normalized MBA relevance.
- `user_item_matrix.npz` for ALS and downstream reproducibility.
- `candidate_set_module2.csv` with unioned, deduplicated, normalized relevance scores.

This notebook is written defensively: if processed parquet artifacts are unavailable (for example due to Git LFS pointers), it reconstructs required tables from raw CSV files.

In [4]:
from pathlib import Path
import importlib
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.sparse import save_npz

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.data_loader as data_loader_module
import src.recall_engine as recall_engine_module

importlib.reload(data_loader_module)
importlib.reload(recall_engine_module)

from src.data_loader import load_or_build_master_transactions
from src.recall_engine import build_als_model, build_candidate_set, build_mba_rules, save_als_factors

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# ---- Reproducibility ----
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- Runtime controls ----
SAMPLE_NROWS = 1_000_000      # Set to None for full corpus run
CANDIDATE_USERS_LIMIT = 500   # None for all users
TOP_ALS = 50
TOP_MBA = 50
SEED_ITEMS_K = 3
TOP_ALS_EXPORT = 100          # Task 2: Keep top-100 ALS scores per user
ALS_SCORE_BATCH_SIZE = 256

# ---- Project paths (robust to notebook launch location) ----
DATA_RAW = PROJECT_ROOT / "data" / "01_raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "02_processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())
print("DATA_PROCESSED exists:", DATA_PROCESSED.exists())

PROJECT_ROOT: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey
DATA_RAW exists: True
DATA_PROCESSED exists: True


In [5]:
# Use a clean processed directory to avoid cached parquet schema issues.
all_path = DATA_PROCESSED / "master_transactions_all.parquet"
organic_path = DATA_PROCESSED / "master_transactions_organic_only.parquet"
if all_path.exists() or organic_path.exists():
    DATA_PROCESSED = DATA_PROCESSED / "notebook_tmp"
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

tx_all, tx_org = load_or_build_master_transactions(
    raw_dir=DATA_RAW,
    processed_dir=DATA_PROCESSED,
    sample_nrows=SAMPLE_NROWS,
)

print("all transactions shape:", tx_all.shape)
print("organic-only transactions shape:", tx_org.shape)
print("households:", tx_all["household_key"].nunique())
print("baskets:", tx_all["BASKET_ID"].nunique())
print("commodities:", tx_all["COMMODITY_DESC"].nunique())
print("day range:", int(tx_all["DAY"].min()), "->", int(tx_all["DAY"].max()))

promo_basket_share = tx_all.groupby("BASKET_ID")["Is_Promoted_Item"].any().mean()
print(f"share of promoted baskets: {promo_basket_share:.3f}")

display(tx_all.head(3))
display(tx_org.head(3))

all transactions shape: (1000000, 13)
organic-only transactions shape: (38852, 13)
households: 2497
baskets: 108962
commodities: 302
day range: 1 -> 318
share of promoted baskets: 0.829


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,COMMODITY_DESC,Is_Promoted_Item,Revenue_Retailer
0,2375,26984851472,1,1004906,1,1.39,0.6,1,0.0,0.0,POTATOES,True,1.39
1,2375,26984851472,1,1033142,1,0.82,0.0,1,0.0,0.0,ONIONS,False,0.82
2,2375,26984851472,1,1036325,1,0.99,0.3,1,0.0,0.0,VEGETABLES - ALL OTHERS,True,0.99


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,COMMODITY_DESC,Is_Promoted_Item,Revenue_Retailer
21,1173,26984945254,1,824399,2,1.98,0.0,1,0.0,0.0,CANDY - PACKAGED,False,1.98
22,1173,26984945254,1,923972,1,0.67,0.0,1,0.0,0.0,CANDY - CHECKLANE,False,0.67
23,1173,26984945254,1,1131351,1,0.88,0.0,1,0.0,0.0,ELECTRICAL SUPPPLIES,False,0.88


In [6]:
mba_rules_long, mba_rules_raw = build_mba_rules(tx_org)
print("Expanded MBA rules:", len(mba_rules_long))
display(mba_rules_long.head(10))

if mba_rules_long.empty:
    print("MBA rules are empty for this sample; downstream candidate generation will rely on ALS only.")

Expanded MBA rules: 588


,antecedent_item,consequent_item,lift,confidence,support,relevance_mba
0,ANALGESICS,COLD AND FLU,7.294737,0.114035,0.001563,1.000000
1,ANALGESICS,WATER - CARBONATED/FLVRD DRINK,1.723166,0.078947,0.001082,0.361583
2,ANALGESICS,CANDY - CHECKLANE,1.247933,0.096491,0.001323,0.123967
3,ANTACIDS,CIGARETTES,2.379777,0.180000,0.001082,0.689889
4,APPLES,CITRUS,6.211765,0.100840,0.001443,1.000000
5,APPLES,TROPICAL FRUIT,5.418685,0.243697,0.003487,1.000000
6,APPLES,SALAD BAR,2.169297,0.117647,0.001684,0.584648
7,APPLES,WATER - CARBONATED/FLVRD DRINK,1.650764,0.075630,0.001082,0.325382
8,BABY FOODS,DIAPERS & DISPOSABLES,10.171699,0.155340,0.001924,1.000000
9,BABY FOODS,BABY HBC,8.548715,0.087379,0.001082,1.000000


In [7]:
(
    als_model,
    user_item_matrix,
    user_to_idx,
    idx_to_user,
    item_to_idx,
    idx_to_item,
    users,
    items,
    user_factors,
    item_factors,
) = build_als_model(tx_all)

user_factor_path, item_factor_path = save_als_factors(
    output_dir=DATA_PROCESSED,
    users=users,
    items=items,
    user_factors=user_factors,
    item_factors=item_factors,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
)

save_npz(DATA_PROCESSED / "user_item_matrix.npz", user_item_matrix)
print("ALS matrix shape (users x items):", user_item_matrix.shape)
print("ALS non-zero interactions:", user_item_matrix.nnz)
print("ALS user factor shape:", user_factors.shape)
print("ALS item factor shape:", item_factors.shape)
print("Saved:")
print(" -", user_factor_path)
print(" -", item_factor_path)
print(" -", DATA_PROCESSED / "user_item_matrix.npz")

  0%|          | 0/20 [00:00<?, ?it/s]

ALS matrix shape (users x items): (2497, 302)
ALS non-zero interactions: 198501
ALS user factor shape: (2497, 64)
ALS item factor shape: (302, 64)
Saved:
 - C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\user_factors.npz
 - C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\item_factors.npz
 - C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\user_item_matrix.npz


In [8]:
(
    als_model,
    user_item_matrix,
    user_to_idx,
    idx_to_user,
    item_to_idx,
    idx_to_item,
    users,
    items,
    user_factors,
    item_factors,
) = build_als_model(tx_all)

user_factor_path, item_factor_path = save_als_factors(
    output_dir=DATA_PROCESSED,
    users=users,
    items=items,
    user_factors=user_factors,
    item_factors=item_factors,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
)

save_npz(DATA_PROCESSED / "user_item_matrix.npz", user_item_matrix)
print("ALS matrix shape (users x items):", user_item_matrix.shape)
print("ALS non-zero interactions:", user_item_matrix.nnz)
print("ALS user factor shape:", user_factors.shape)
print("ALS item factor shape:", item_factors.shape)
print("Saved:")
print(" -", user_factor_path)
print(" -", item_factor_path)
print(" -", DATA_PROCESSED / "user_item_matrix.npz")

  0%|          | 0/20 [00:00<?, ?it/s]

ALS matrix shape (users x items): (2497, 302)
ALS non-zero interactions: 198501
ALS user factor shape: (2497, 64)
ALS item factor shape: (302, 64)
Saved:
 - C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\user_factors.npz
 - C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\item_factors.npz
 - C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\user_item_matrix.npz


In [9]:
# Build the recall-layer artifacts from the shared src implementation.
artifacts = build_candidate_set(
    tx_all=tx_all,
    mba_rules_long=mba_rules_long,
    users=users,
    user_to_idx=user_to_idx,
    user_factors=user_factors,
    item_factors=item_factors,
    items=items,
    top_als=TOP_ALS,
    top_mba=TOP_MBA,
    seed_items_k=SEED_ITEMS_K,
    candidate_users_limit=CANDIDATE_USERS_LIMIT,
    als_score_batch_size=ALS_SCORE_BATCH_SIZE,
)

candidate_set = artifacts.candidate_set
filtered_items_log = artifacts.filtered_items_log
als_scores = artifacts.als_scores
seed_summary = artifacts.seed_summary
recent_pairs = artifacts.recent_pairs

als_scores_path = DATA_PROCESSED / "als_scores.parquet"
als_scores.to_parquet(als_scores_path, index=False)

mba_rules_long.to_csv(DATA_PROCESSED / "association_rules.csv", index=False)
candidate_set.to_csv(DATA_PROCESSED / "candidate_set_module2.csv", index=False)
filtered_items_log.to_csv(DATA_PROCESSED / "filtered_items_log.csv", index=False)

print("ALS top-100 score rows:", len(als_scores))
print("Saved:", als_scores_path)
print("Saved:", DATA_PROCESSED / "association_rules.csv")
print("Saved:", DATA_PROCESSED / "candidate_set_module2.csv")
print("Saved:", DATA_PROCESSED / "filtered_items_log.csv")
print("users selected:", candidate_set["household_key"].nunique())
print("candidate rows:", len(candidate_set))
print("avg candidates per user:", round(candidate_set.groupby("household_key").size().mean(), 2))
print("source mix:")
print(candidate_set["source_detail"].value_counts(normalize=True).round(3))

display(candidate_set.head(20))
display(filtered_items_log.head(20))

ALS top-100 score rows: 25000
Saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\als_scores.parquet
Saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\association_rules.csv
Saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\candidate_set_module2.csv
Saved: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\notebook_tmp\filtered_items_log.csv
users selected: 500
candidate rows: 22786
avg candidates per user: 45.57
source mix:
source_detail
ALS        0.674
MBA        0.179
BOTH       0.131
UNKNOWN    0.016
Name: proportion, dtype: float64


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
3,1,BAKED SWEET GOODS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.817462,0.000000,0.817462,als,ALS,46,False
5,1,BATH TISSUES,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.084018,0.000000,0.084018,als,ALS,46,False
6,1,BEANS - CANNED GLASS & MW,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.272483,0.000000,0.272483,als,ALS,46,False
7,1,BEEF,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.000000,0.798726,0.798726,mba,MBA,46,False
8,1,BEERS/ALES,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.000000,0.826111,0.826111,mba,MBA,46,False
10,1,BREAKFAST SWEETS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.837654,0.022219,0.837654,both,BOTH,46,False
11,1,CANDY - CHECKLANE,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.013540,0.000000,0.013540,als,ALS,46,False
13,1,CARROTS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.900266,0.000000,0.900266,als,ALS,46,False
15,1,CHEESES,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.487200,0.000000,0.487200,als,ALS,46,False
16,1,COFFEE,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.446381,0.000000,0.446381,als,ALS,46,False


,household_key,COMMODITY_DESC,filter_reason,reference_week
0,1,APPLES,recent_purchase,46
1,1,BAG SNACKS,recent_purchase,46
2,1,BAKED BREAD/BUNS/ROLLS,recent_purchase,46
4,1,BAKING MIXES,recent_purchase,46
9,1,BLEACH,recent_purchase,46
12,1,CANDY - PACKAGED,recent_purchase,46
14,1,CHEESE,recent_purchase,46
17,1,COLD CEREAL,recent_purchase,46
18,1,CONDIMENTS/SAUCES,recent_purchase,46
19,1,COOKIES/CONES,recent_purchase,46


In [10]:
if candidate_set.empty:
    print("Candidate set is empty after filtering.")
else:
    source_pct = candidate_set["source_detail"].value_counts(normalize=True).mul(100).round(2)
    print("Source composition (%):")
    for key in ["ALS", "MBA", "BOTH"]:
        print(f" - {key}: {source_pct.get(key, 0.0)}%")

    print("\nCandidate count per user quantiles:")
    coverage = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique()
    print(coverage.quantile([0.0, 0.25, 0.5, 0.75, 1.0]).round(2))

    print("\nSample top-5 recommendations for 3 users:")
    for hh in candidate_set["household_key"].drop_duplicates().head(3):
        print(f"\nHousehold {hh}")
        display(candidate_set[candidate_set["household_key"] == hh].sort_values("Relevance", ascending=False).head(5))

Source composition (%):
 - ALS: 67.43%
 - MBA: 17.9%
 - BOTH: 13.1%

Candidate count per user quantiles:
0.00     9.0
0.25    35.0
0.50    46.0
0.75    55.0
1.00    77.0
Name: COMMODITY_DESC, dtype: float64

Sample top-5 recommendations for 3 users:

Household 1


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
22,1,DINNER SAUSAGE,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.000000,1.0,1.0,mba,MBA,46,False
43,1,PNT BTR/JELLY/JAMS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.884771,1.0,1.0,both,BOTH,46,False
58,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.483568,1.0,1.0,both,BOTH,46,False
56,1,TOMATOES,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.000000,1.0,1.0,mba,MBA,46,False
39,1,ONIONS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.367548,1.0,1.0,both,BOTH,46,False



Household 2


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
68,2,BATH TISSUES,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,1.000000,0.649651,1.000000,both,BOTH,46,False
98,2,FUEL,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba,MBA,46,False
81,2,COOKIES/CONES,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba,MBA,46,False
131,2,TOBACCO OTHER,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba,MBA,46,False
136,2,YOGURT,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.984508,0.000000,0.984508,als,ALS,46,False



Household 3


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
155,3,DINNER MXS:DRY,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,1.000000,0.0,1.0,als,ALS,46,False
159,3,DRY SAUCES/GRAVY,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.000000,1.0,1.0,mba,MBA,46,False
157,3,DRY BN/VEG/POTATO/RICE,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.678581,1.0,1.0,both,BOTH,46,False
192,3,VEGETABLES SALAD,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.020895,1.0,1.0,both,BOTH,46,False
178,3,PORK,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.034267,1.0,1.0,both,BOTH,46,False


In [11]:
if not candidate_set.empty:
    avg_candidates = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique().mean()
    print("1) Avg candidates per user:", round(float(avg_candidates), 2), "(target: 100-200)")

    source_pct = candidate_set["source_detail"].value_counts(normalize=True).mul(100).round(2)
    print("\n2) Source composition (%):")
    for key in ["ALS", "MBA", "BOTH"]:
        print(f" - {key}: {source_pct.get(key, 0.0)}%")

    removed_count = len(filtered_items_log)
    pre_filter_count = len(candidate_set) + len(filtered_items_log)
    removed_pct = (removed_count / pre_filter_count * 100.0) if pre_filter_count > 0 else 0.0
    print("\n3) Removed due to recency filter:", round(removed_pct, 2), "%")

    leak_check = candidate_set.merge(
        recent_pairs[["household_key", "COMMODITY_DESC"]],
        on=["household_key", "COMMODITY_DESC"],
        how="inner",
    )
    print("\n4) Leakage check (must be 0):", len(leak_check))

    print("\nCandidate count per user quantiles:")
    coverage = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique()
    print(coverage.quantile([0.0, 0.25, 0.5, 0.75, 1.0]).round(2))

    for column in ["relevance_als", "relevance_mba", "Relevance"]:
        low_ok = bool((candidate_set[column] >= 0.0).all())
        high_ok = bool((candidate_set[column] <= 1.0).all())
        print(f"{column}: within [0,1] ->", low_ok and high_ok)

    print("\nSample top-5 recommendations for 3 users:")
    for hh in candidate_set["household_key"].drop_duplicates().head(3):
        print(f"\nHousehold {hh}")
        display(candidate_set[candidate_set["household_key"] == hh].sort_values("Relevance", ascending=False).head(5))

1) Avg candidates per user: 45.57 (target: 100-200)

2) Source composition (%):
 - ALS: 67.43%
 - MBA: 17.9%
 - BOTH: 13.1%

3) Removed due to recency filter: 22.83 %

4) Leakage check (must be 0): 0

Candidate count per user quantiles:
0.00     9.0
0.25    35.0
0.50    46.0
0.75    55.0
1.00    77.0
Name: COMMODITY_DESC, dtype: float64
relevance_als: within [0,1] -> True
relevance_mba: within [0,1] -> True
Relevance: within [0,1] -> True

Sample top-5 recommendations for 3 users:

Household 1


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
22,1,DINNER SAUSAGE,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.000000,1.0,1.0,mba,MBA,46,False
43,1,PNT BTR/JELLY/JAMS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.884771,1.0,1.0,both,BOTH,46,False
58,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.483568,1.0,1.0,both,BOTH,46,False
56,1,TOMATOES,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.000000,1.0,1.0,mba,MBA,46,False
39,1,ONIONS,BAKED BREAD/BUNS/ROLLS | SNACK NUTS | CITRUS,0.367548,1.0,1.0,both,BOTH,46,False



Household 2


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
68,2,BATH TISSUES,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,1.000000,0.649651,1.000000,both,BOTH,46,False
98,2,FUEL,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba,MBA,46,False
81,2,COOKIES/CONES,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba,MBA,46,False
131,2,TOBACCO OTHER,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba,MBA,46,False
136,2,YOGURT,ICE CREAM/MILK/SHERBTS | SOFT DRINKS | CIGARETTES,0.984508,0.000000,0.984508,als,ALS,46,False



Household 3


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased
155,3,DINNER MXS:DRY,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,1.000000,0.0,1.0,als,ALS,46,False
159,3,DRY SAUCES/GRAVY,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.000000,1.0,1.0,mba,MBA,46,False
157,3,DRY BN/VEG/POTATO/RICE,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.678581,1.0,1.0,both,BOTH,46,False
192,3,VEGETABLES SALAD,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.020895,1.0,1.0,both,BOTH,46,False
178,3,PORK,VEGETABLES - SHELF STABLE | REFRGRATD DOUGH PR...,0.034267,1.0,1.0,both,BOTH,46,False


## Handoff to Module 3 (Utility Ranking)

This notebook now exports a reproducible and explainable Module 2 artifact set:

- `association_rules.csv`: normalized MBA signal (`relevance_mba`).
- `als_scores.parquet`: top-100 ALS dot-product scores per user, normalized to `[0,1]`.
- `user_factors.npz`: ALS user latent factors + `user_id_to_index` mapping metadata.
- `item_factors.npz`: ALS item latent factors + `item_id_to_index` mapping metadata.
- `candidate_set_module2.csv`: recall-union candidates with source traceability fields.
- `filtered_items_log.csv`: explicit audit log of recency-based removals.
- `user_item_matrix.npz`: sparse confidence matrix cache.

Candidate set columns include:
- `relevance_als`
- `relevance_mba`
- `Relevance = max(relevance_als, relevance_mba)`
- `source_detail` in `{ALS, MBA, BOTH}`
- `snapshot_week`
- `was_recently_purchased`

Module 3 can directly consume `candidate_set_module2.csv` and combine with margin, uplift, and context features.

$$
U(i,u)=w_1\cdot Relevance(i,u)+w_2\cdot Uplift(i,u)+w_3\cdot Margin(i)+w_4\cdot Context(i,u)
$$

with baseline weights:
- $w_1=0.40$
- $w_2=0.25$
- $w_3=0.20$
- $w_4=0.15$.